# Naija 航空公司价格查找器

## 练习目标

用 **Playwright 爬虫**（同目录 `scraper.py`）抓取 **Travelwings** 动态页面票价，再通过 **OpenAI Function Calling** 把「出发地 / 目的地 / 日期」解析成工具参数，最后用 **Nigerian Pidgin** 风格回复。

## 和本课的关系

| 概念 | 本笔记本 |
|------|----------|
| Tool Use | 工具名 `check_travelwings` → 实际调用 `scrape_travelwings_prices` |
| 外部副作用 | 浏览器自动化爬取（可能较慢 10–15 秒） |
| Gradio UI | `ChatInterface` + `share=True` |

## 怎么跑

1. 安装依赖并准备 Playwright；`.env` 有 `OPENAI_API_KEY`
2. 确保同目录有可导入的 `scraper.py`
3. 运行全部单元格，浏览器打开 Gradio；用示例问法试查价


In [2]:
# ========== 导入：环境、JSON、Gradio、OpenAI、本地爬虫 ==========

# 标准库 os：读 OPENAI_API_KEY
import os
# 标准库 json：解析 tool_call.function.arguments
import json
# gradio：ChatInterface 聊天 UI
import gradio as gr
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI 官方客户端
from openai import OpenAI
# 同目录 scraper.py：Playwright 抓 Travelwings 价格文本
from scraper import scrape_travelwings_prices


ModuleNotFoundError: No module named 'playwright'

In [ ]:
# ========== 加载密钥并创建 OpenAI 客户端 ==========

# 把 .env 读入环境变量（确保 OPENAI_API_KEY 可用）
load_dotenv()
# 取出 API Key
api_key = os.getenv('OPENAI_API_KEY')

# 缺密钥时打印 Pidgin 风格提示；有密钥则确认（文案保持原样）
if not api_key:
    print("Ah ah! No API key was found - please make sure it's set in your environment variables!")
else:
    print("API key found and we are good to go!")

# 用密钥创建客户端，供后面 chat.completions.create 使用
client = OpenAI(api_key=api_key)


In [ ]:
# ========== 系统提示 + 工具 schema + 带工具循环的聊天处理 ==========

# system_prompt：规定 Pidgin 口吻、必须调用工具、如何读爬取结果——英文正文勿改
system_prompt = """
You are the "Naija Ticket Master" — a very sharp Nigerian travel agent.

Your job is to read the user request, figure out the Origin, Destination, and Travel Date, and then USE YOUR `check_travelwings` TOOL.
Note: The tool requires 3-letter IATA airport codes (e.g., Abuja = ABV, Lagos = LOS) and dates in YYYY-MM-DD format.

Your Rules:
1. Speak completely in fun, relatable Nigerian Pidgin English.
2. Always call the `check_travelwings` tool when a user asks for a flight.
3. Wait for the tool output, which will give you the scraped text from Travelwings website.
4. Read the scraped text carefully. Look for airlines (e.g., Air Peace, APG Airlines) and their exact prices (e.g., NGN 92,611).
5. List out the available airlines and their prices using bullet points. Mention if they are Economy, Business etc.
6. If no prices are found, say so.
7. End with a funny one-liner about traveling in Nigeria.
"""

# tools：只暴露一个函数 check_travelwings；参数为 IATA + YYYY-MM-DD
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_travelwings",
            "description": "Scrapes Travelwings flight data for a specific route and returns unstructured text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {
                        "type": "string",
                        "description": "The 3-letter IATA code for the departure city (e.g., ABV for Abuja)."
                    },
                    "destination": {
                        "type": "string",
                        "description": "The 3-letter IATA code for the arrival city (e.g., LOS for Lagos)."
                    },
                    "date": {
                        "type": "string",
                        "description": "The date of travel in YYYY-MM-DD format."
                    }
                },
                "required": ["origin", "destination", "date"],
                # 禁止额外字段，强制模型只填这三个参数
                "additionalProperties": False,
            },
        }
    }
]

# Gradio 回调：user_message 本轮话，history 是 [(user, assistant), ...]
def handle_chat(user_message: str, history) -> str:
    # 每轮从 system 开始重建 messages
    messages = [{"role": "system", "content": system_prompt}]

    # 把 Gradio 历史成对还原成 user/assistant 消息
    for past_user, past_assistant in history:
        messages.append({"role": "user", "content": past_user})
        messages.append({"role": "assistant", "content": past_assistant})

    # 追加当前用户输入
    messages.append({"role": "user", "content": user_message})

    try:
        # 可能多轮工具调用，直到模型给出最终文本
        while True:
            # 请求模型；model / temperature / tools 参数保持原样
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools,
                temperature=0.7
            )

            # 取出 assistant 消息
            response_message = response.choices[0].message

            # 若要求调用工具
            if response_message.tool_calls:
                # 先把含 tool_calls 的 assistant 消息入列
                messages.append(response_message)

                for tool_call in response_message.tool_calls:
                    if tool_call.function.name == "check_travelwings":
                        # 解析 JSON 参数
                        args = json.loads(tool_call.function.arguments)
                        # 调试打印：看模型解析出的 origin/destination/date
                        print(f"Executing Playwright Scraper for: {args}")

                        origin = args.get("origin")
                        destination = args.get("destination")
                        date_val = args.get("date")

                        # 真正跑 Playwright 爬虫（可能 10–15 秒）
                        function_result = scrape_travelwings_prices(origin, destination, date_val)

                        # 以 role=tool 把非结构化网页文本交回模型
                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": function_result
                        })
                # while 继续：带着工具结果再问模型
            else:
                # 无工具调用 → 最终 Pidgin 回复
                return response_message.content

    except Exception as e:
        # 异常时的 Pidgin 错误文案保持原样
        return f"Omo, wahala dey! Error don occur: {str(e)}"


In [ ]:
# ========== 启动 Gradio ChatInterface ==========

# fn 绑定 handle_chat；title/description/examples 为 UI 文案，保持英文原样
demo = gr.ChatInterface(
    fn=handle_chat,
    title="Naija Airline Ticket Master ✈️💸",
    description="Just tell me where you wanna go and when! (e.g. 'Abuja to Lagos on 2026-03-03')",
    examples=[
        "Check flight prices from Abuja to Lagos on 2026-03-03",
        "Any Kano to Abuja flights for next week?"
    ]
)

# share=True：生成可分享的临时公网链接（原参数保留）
demo.launch(share=True)
